In [0]:
# ====================================================================
# ML MODEL 1: CUSTOMER CHURN PREDICTION
# ====================================================================
# Purpose: Predict which customers are likely to churn (stop buying)
#          using RFM scores and purchase behavior patterns
# ====================================================================

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, roc_curve
)
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns

# Configuration
PROJECT_NAME = "retail"
CATALOG = "workspace"
SILVER_SCHEMA = f"{PROJECT_NAME}_silver"
GOLD_SCHEMA = f"{PROJECT_NAME}_gold"

# MLflow experiment name
EXPERIMENT_NAME = f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/retail-churn-prediction"

print("=" * 80)
print("🤖 CUSTOMER CHURN PREDICTION MODEL")
print("=" * 80)
print(f"Experiment: {EXPERIMENT_NAME}")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# MLFLOW EXPERIMENT SETUP
# ====================================================================
print("🔬 Setting up MLflow experiment...\n")

# Set or create experiment
mlflow.set_experiment(EXPERIMENT_NAME)

# Get experiment details
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
print(f"✅ Experiment ID: {experiment.experiment_id}")
print(f"✅ Experiment Location: {experiment.artifact_location}")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# LOAD TRAINING DATA
# ====================================================================
print("📂 Loading customer data...\n")

# Load RFM scores (has customer features)
rfm_df = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.gold_rfm_scores")

# Load customer master (has additional features)
customers_df = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_customers_master")

print(f"✅ RFM scores: {rfm_df.count():,} customers")
print(f"✅ Customer master: {customers_df.count():,} customers")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# DEFINE CHURN LABEL
# ====================================================================
print("🏷️ Defining churn labels...\n")

# Join RFM with customer master to get all features
ml_dataset = (rfm_df
    .join(
        customers_df.select(
            "customer_id",
            "avg_delivery_days",
            "late_delivery_rate",
            "unique_categories_purchased",
            "unique_sellers_purchased_from",
            "customer_lifetime_days"
        ),
        "customer_id",
        "inner"
    )
)

# Define churn: customer hasn't ordered in 90+ days
# Convert to Pandas for sklearn
ml_df = ml_dataset.toPandas()

# Create churn label (1 = churned, 0 = active)
CHURN_THRESHOLD_DAYS = 90
ml_df['is_churned'] = (ml_df['days_since_last_order'] >= CHURN_THRESHOLD_DAYS).astype(int)

print(f"Total customers: {len(ml_df):,}")
print(f"Churned customers: {ml_df['is_churned'].sum():,} ({ml_df['is_churned'].mean()*100:.1f}%)")
print(f"Active customers: {(1-ml_df['is_churned']).sum():,} ({(1-ml_df['is_churned'].mean())*100:.1f}%)")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# FEATURE ENGINEERING
# ====================================================================
print("🔧 Engineering features...\n")

# Select features for the model
feature_columns = [
    # RFM scores
    'recency_score',
    'frequency_score',
    'monetary_score',
    'rfm_total_score',
    
    # Purchase behavior
    'total_orders',
    'total_spent',
    'days_since_last_order',
    'customer_lifetime_days',
    
    # Product diversity
    'unique_categories_purchased',
    'unique_sellers_purchased_from',
    
    # Delivery experience
    'avg_delivery_days',
    'late_delivery_rate',
    
    # Flags
    'is_high_value_customer',
    'is_repeat_customer'
]

# Convert boolean columns to int
ml_df['is_high_value_customer'] = ml_df['is_high_value_customer'].astype(int)
ml_df['is_repeat_customer'] = ml_df['is_repeat_customer'].astype(int)

# Handle any missing values
ml_df[feature_columns] = ml_df[feature_columns].fillna(0)

# Prepare X (features) and y (target)
X = ml_df[feature_columns]
y = ml_df['is_churned']

print(f"Features: {len(feature_columns)}")
print(f"Feature list:")
for i, feat in enumerate(feature_columns, 1):
    print(f"  {i}. {feat}")

print("\nFeature statistics:")
print(X.describe())
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# TRAIN/TEST SPLIT
# ====================================================================
print("✂️ Splitting data into train/test sets...\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # Maintain class balance
)

print(f"Training set: {len(X_train):,} samples")
print(f"  - Churned: {y_train.sum():,} ({y_train.mean()*100:.1f}%)")
print(f"  - Active: {(1-y_train).sum():,} ({(1-y_train.mean())*100:.1f}%)")

print(f"\nTest set: {len(X_test):,} samples")
print(f"  - Churned: {y_test.sum():,} ({y_test.mean()*100:.1f}%)")
print(f"  - Active: {(1-y_test).sum():,} ({(1-y_test.mean())*100:.1f}%)")

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# FEATURE SCALING
# ====================================================================
print("📏 Scaling features...\n")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled using StandardScaler")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# MODEL 1: RANDOM FOREST CLASSIFIER
# ====================================================================
print("🌲 Training Random Forest Classifier...\n")

# Start MLflow run
with mlflow.start_run(run_name="random_forest_churn") as run:
    
    # Log parameters
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    mlflow.log_param("churn_threshold_days", CHURN_THRESHOLD_DAYS)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("features", feature_columns)
    
    # Train model
    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    
    rf_model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred = rf_model.predict(X_test_scaled)
    y_pred_proba = rf_model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    
    # Print results
    print("✅ Random Forest Results:")
    print(f"  - Accuracy:  {accuracy:.4f}")
    print(f"  - Precision: {precision:.4f}")
    print(f"  - Recall:    {recall:.4f}")
    print(f"  - F1 Score:  {f1:.4f}")
    print(f"  - ROC AUC:   {roc_auc:.4f}")
    
    print("\n📊 Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))
    
    # Log model with signature
    signature = infer_signature(X_train_scaled, rf_model.predict(X_train_scaled))
    mlflow.sklearn.log_model(
        rf_model, 
        "random_forest_model",
        signature=signature,
        registered_model_name="retail_churn_random_forest"
    )
    
    rf_run_id = run.info.run_id
    print(f"\n✅ Model logged to MLflow (Run ID: {rf_run_id})")

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# MODEL 2: GRADIENT BOOSTING CLASSIFIER
# ====================================================================
print("🚀 Training Gradient Boosting Classifier...\n")

with mlflow.start_run(run_name="gradient_boosting_churn") as run:
    
    # Log parameters
    mlflow.log_param("model_type", "GradientBoostingClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("churn_threshold_days", CHURN_THRESHOLD_DAYS)
    mlflow.log_param("test_size", 0.2)
    
    # Train model
    gb_model = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    )
    
    gb_model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred = gb_model.predict(X_test_scaled)
    y_pred_proba = gb_model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    
    # Print results
    print("✅ Gradient Boosting Results:")
    print(f"  - Accuracy:  {accuracy:.4f}")
    print(f"  - Precision: {precision:.4f}")
    print(f"  - Recall:    {recall:.4f}")
    print(f"  - F1 Score:  {f1:.4f}")
    print(f"  - ROC AUC:   {roc_auc:.4f}")
    
    print("\n📊 Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))
    
    # Log model
    signature = infer_signature(X_train_scaled, gb_model.predict(X_train_scaled))
    mlflow.sklearn.log_model(
        gb_model, 
        "gradient_boosting_model",
        signature=signature,
        registered_model_name="retail_churn_gradient_boosting"
    )
    
    gb_run_id = run.info.run_id
    print(f"\n✅ Model logged to MLflow (Run ID: {gb_run_id})")

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# FEATURE IMPORTANCE ANALYSIS
# ====================================================================
print("📊 Analyzing feature importance...\n")

# Get feature importance from Random Forest
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'][:10], feature_importance['importance'][:10])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importance (Random Forest)')
plt.gca().invert_yaxis()
plt.tight_layout()

# Save plot
plt.savefig('/tmp/feature_importance.png', dpi=150, bbox_inches='tight')
print("\n✅ Feature importance plot saved")

# Display plot
display(plt.gcf())
plt.close()

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# CONFUSION MATRIX
# ====================================================================
print("📉 Creating confusion matrix...\n")

# Use Gradient Boosting predictions
y_pred_gb = gb_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred_gb)

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Active', 'Churned'],
            yticklabels=['Active', 'Churned'])
plt.title('Confusion Matrix - Gradient Boosting Model')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()

# Save and display
plt.savefig('/tmp/confusion_matrix.png', dpi=150, bbox_inches='tight')
print("✅ Confusion matrix saved")
display(plt.gcf())
plt.close()

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# ROC CURVE
# ====================================================================
print("📈 Creating ROC curve...\n")

y_pred_proba_gb = gb_model.predict_proba(X_test_scaled)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_gb)
roc_auc = roc_auc_score(y_test, y_pred_proba_gb)

# Plot
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Churn Prediction Model')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()

# Save and display
plt.savefig('/tmp/roc_curve.png', dpi=150, bbox_inches='tight')
print("✅ ROC curve saved")
display(plt.gcf())
plt.close()

print("=" * 80 + "\n")

In [0]:
# ====================================================================
# SCORE ALL CUSTOMERS
# ====================================================================
print("🎯 Scoring all customers with churn probability...\n")

# Scale all features
X_all_scaled = scaler.transform(X)

# Predict churn probability for all customers
churn_probabilities = gb_model.predict_proba(X_all_scaled)[:, 1]

# Add predictions to dataframe
ml_df['churn_probability'] = churn_probabilities
ml_df['churn_prediction'] = (churn_probabilities >= 0.5).astype(int)

# Create risk segments
ml_df['churn_risk_segment'] = pd.cut(
    churn_probabilities,
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

print("Churn Risk Distribution:")
print(ml_df['churn_risk_segment'].value_counts().sort_index())

print("\n✅ All customers scored")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# SAVE PREDICTIONS
# ====================================================================
print("💾 Saving churn predictions...\n")

# Select relevant columns
predictions_df = ml_df[[
    'customer_id',
    'customer_city',
    'customer_state',
    'rfm_segment',
    'total_orders',
    'total_spent',
    'days_since_last_order',
    'is_churned',
    'churn_probability',
    'churn_prediction',
    'churn_risk_segment'
]]

# Convert to Spark DataFrame
predictions_spark_df = spark.createDataFrame(predictions_df)

# Write to gold schema
predictions_table = f"{CATALOG}.{GOLD_SCHEMA}.gold_churn_predictions"

(predictions_spark_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(predictions_table))

print(f"✅ Saved predictions to: {predictions_table}")
print(f"   Total customers: {predictions_df.shape[0]:,}")
print("=" * 80 + "\n")

In [0]:
print("\n" + "=" * 80)
print("🎉 CHURN PREDICTION MODEL COMPLETE")
print("=" * 80)

print("\n📊 Model Performance Summary:")
print("  Random Forest:")
print(f"    - Accuracy: {accuracy_score(y_test, rf_model.predict(X_test_scaled)):.4f}")
print(f"    - ROC AUC: {roc_auc_score(y_test, rf_model.predict_proba(X_test_scaled)[:, 1]):.4f}")

print("\n  Gradient Boosting:")
print(f"    - Accuracy: {accuracy_score(y_test, gb_model.predict(X_test_scaled)):.4f}")
print(f"    - ROC AUC: {roc_auc_score(y_test, gb_model.predict_proba(X_test_scaled)[:, 1]):.4f}")

print("\n📁 Outputs Created:")
print(f"  - MLflow Experiment: {EXPERIMENT_NAME}")
print(f"  - Registered Models: retail_churn_random_forest, retail_churn_gradient_boosting")
print(f"  - Predictions Table: {predictions_table}")

print("\n🔍 High Risk Customers:")
high_risk = ml_df[ml_df['churn_risk_segment'] == 'High Risk']
print(f"  - Count: {len(high_risk):,}")
print(f"  - Total Revenue at Risk: ${high_risk['total_spent'].sum():,.2f}")

print("\n📝 Next Steps:")
print("  1. Review models in MLflow UI (Machine Learning → Experiments)")
print("  2. Query predictions: SELECT * FROM gold_churn_predictions WHERE churn_risk_segment = 'High Risk'")
print("  3. Run 02_demand_forecast.py for demand forecasting model")

print("=" * 80 + "\n")

In [0]:
%sql
-- High risk churn customers
SELECT 
    customer_id,
    customer_city,
    customer_state,
    rfm_segment,
    total_orders,
    ROUND(total_spent, 2) as total_spent,
    days_since_last_order,
    ROUND(churn_probability * 100, 2) as churn_probability_pct,
    churn_risk_segment
FROM workspace.retail_gold.gold_churn_predictions
WHERE churn_risk_segment = 'High Risk'
ORDER BY churn_probability DESC
LIMIT 20;